In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

import hashlib
import pickle
from pathlib import Path

from sklearn.neural_network import MLPRegressor

from model import single_ml_model_exp, grid_search_exp

import config
%load_ext autoreload
%autoreload 2

In [ ]:
# === Notebook ONE-OFF -- baseline MLP single SO para as 2 series novas ===
# (windspeedfortaleza, samurec), adicionadas em 2026-08-28.
#
# NAO e o notebook de baseline protegido de 17 series -- nao importa, nao
# referencia, nao compartilha lista com ele (RUNBOOK.md Secao 6 / CLAUDE.md
# Secao 3: os notebooks de baseline protegidos nunca devem ser editados nem
# re-rodados com escopo alterado). Este arquivo replica a MESMA logica
# (mesmos hiperparametros, model_exec=10, mesma chamada de GridSearch),
# restrita a 2 series e com force=False.
model = MLPRegressor(activation='logistic', solver='lbfgs')

experiment_id = 'chamados'
model_name = 'mlp'
normalize = True
force = False          # OBRIGATORIO: nunca True aqui. Append idempotente --
                       # jamais regenerar os baselines ja validados.
model_exec = 10

experiment_params = {
    'diff_kpss': False,
    'horizon': 1,
    'type_filter': None,
}

model_parameters = {
    'hidden_layer_sizes': [10, 20, 50],
    'max_iter': [1000],
}

# Lista fixa e isolada -- SO as 2 series novas. Deliberadamente NAO lida de
# config.BASE_NAME_LIST (evita acoplamento e surpresa se a lista mudar).
new_series_list = ['windspeedfortaleza.txt', 'samurec.txt']

# --- Snapshot de hash ANTES: TODOS os .pkl de chamados/ (nao so os *_1mlp) ---
# Ao final (celula seguinte) isto confirma automaticamente que este one-off
# nao tocou NENHUM baseline pre-existente -- ARIMA/MLP/SVR/ARIMA-MLP/ARIMA-SVR
# de nenhuma serie, incluindo os das 4 series antigas.
chamados_dir = Path(config.ROOT_PATH) / 'data' / 'result' / 'chamados'
hashes_before = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(chamados_dir.glob('*.pkl'))
}
print(f"snapshot: {len(hashes_before)} .pkl em chamados/ antes da execucao")

for base_name in new_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        single_ml_model_exp.SKlearnModel,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

In [ ]:
# === VERIFICACAO POS-EXECUCAO (versao 2 series, snapshot in-notebook) ===
# Adaptada da celula de verificacao dos notebooks de baseline de 17 series
# (que espera 17 series + um arquivo de snapshot de hash especifico -- nao se
# aplica aqui). Garantia essencial preservada: confirma que NENHUM baseline
# pre-existente foi tocado.
assert len(hashes_before) > 0, (
    "hashes_before vazio -- a celula de configuracao nao rodou, ou chamados/ "
    "estava vazio. Sem baselines pre-existentes para comparar, a checagem (b) "
    "passaria trivialmente sem valor. Rode a celula anterior primeiro."
)

expected_new = {'samurec_1mlp.pkl', 'windspeedfortaleza_1mlp.pkl'}

hashes_after = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(chamados_dir.glob('*.pkl'))
}

created = set(hashes_after) - set(hashes_before)
touched_existing = {n for n in hashes_before if hashes_after.get(n) != hashes_before[n]}

print("--- (a) arquivos criados ---")
print(f"  criados: {sorted(created)}")
assert created == expected_new, (
    f"esperava criar exatamente {sorted(expected_new)}, criou {sorted(created)}"
)

print("--- (b) nenhum .pkl pre-existente foi tocado (TODOS os baselines, nao so MLP) ---")
print(f"  pre-existentes alterados: {sorted(touched_existing)}")
assert not touched_existing, (
    f"ALERTA: {sorted(touched_existing)} mudaram de hash -- este one-off (force=False) "
    "NAO deveria tocar nenhum baseline pre-existente. Investigar antes de prosseguir."
)

print("--- (c) config persistida nos 2 .pkl novos ---")
for name in sorted(expected_new):
    with open(chamados_dir / name, 'rb') as f:
        saved = pickle.load(f)
    obj = saved[0]['experiment']
    dk = obj.experiment_params['diff_kpss']
    print(f"  {name}: diff_kpss={dk!r}")
    assert dk is False, f"{name}: esperado diff_kpss=False, achou {dk!r}"

print()
print("OK -- 2 baselines novos criados; 0 baseline protegido tocado.")